In [2]:
!git clone https://github.com/nathaniel-starlink/CNN-Implementation.git
%cd CNN-Implementation/algoritma_cnn

fatal: destination path 'CNN-Implementation' already exists and is not an empty directory.
/content/CNN-Implementation/algoritma_cnn


In [3]:
import os
import shutil
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from google.colab import files

# 1. Arsitektur Modified LeNet-5 (Input 1x64x64 -> 2 Kelas)
class ModifiedLeNet5(nn.Module):
    def __init__(self, num_classes=2):
        super(ModifiedLeNet5, self).__init__()
        self.conv1 = nn.Conv2d(1, 8, kernel_size=5, stride=1, padding=0, bias=True)   # -> 8x60x60
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)                            # -> 8x30x30
        self.conv2 = nn.Conv2d(8, 16, kernel_size=5, stride=1, padding=0, bias=True)  # -> 16x26x26
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)                            # -> 16x13x13
        self.conv3 = nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=0, bias=True) # -> 32x11x11
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)                            # -> 32x5x5 (800)

        self.relu = nn.ReLU()
        self.fc1 = nn.Linear(32 * 5 * 5, 64, bias=True)
        self.fc2 = nn.Linear(64, num_classes, bias=True)

    def forward(self, x):
        x = self.pool1(self.relu(self.conv1(x)))
        x = self.pool2(self.relu(self.conv2(x)))
        x = self.pool3(self.relu(self.conv3(x)))
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# 2. Fungsi Kuantisasi INT8 & Ekspor ke .mif (Quartus) dan .hex (Questa/ModelSim)
def export_to_mif_and_hex(tensor_float, layer_name, out_dir="../verilog/weights_mif"):
    os.makedirs(out_dir, exist_ok=True)
    arr = tensor_float.detach().cpu().numpy().flatten()

    max_val = np.max(np.abs(arr))
    scale = 127.0 / max_val if max_val > 0 else 1.0
    arr_int8 = np.clip(np.round(arr * scale), -127, 127).astype(np.int8)

    depth = len(arr_int8)
    mif_path = os.path.join(out_dir, f"{layer_name}.mif")
    hex_path = os.path.join(out_dir, f"{layer_name}.hex")

    with open(mif_path, "w") as fmif, open(hex_path, "w") as fhex:
        fmif.write(f"-- Quantized INT8 Weights for {layer_name} (Scale = {scale:.6f})\n")
        fmif.write(f"WIDTH=8;\nDEPTH={depth};\nADDRESS_RADIX=UNS;\nDATA_RADIX=HEX;\nCONTENT BEGIN\n")
        for addr, val in enumerate(arr_int8):
            hex_val = f"{(int(val) & 0xFF):02X}"
            fmif.write(f"  {addr} : {hex_val};\n")
            fhex.write(f"{hex_val}\n")
        fmif.write("END;\n")
    print(f"Exported {layer_name:15s} | Params: {depth:6d} | Scale: {scale:.4f}")

# 3. Persiapan Data & Training
data_dir = "../dataset/processed_64x64"
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor()
])

train_data = datasets.ImageFolder(os.path.join(data_dir, "train"), transform=transform)
val_data = datasets.ImageFolder(os.path.join(data_dir, "valid"), transform=transform)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)

print("Mapping Kelas:", train_data.class_to_idx)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ModifiedLeNet5(num_classes=len(train_data.classes)).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 25
for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            preds = model(imgs).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    acc = 100.0 * correct / max(1, total)
    print(f"Epoch [{epoch+1:02d}/{epochs}] Loss: {total_loss/len(train_loader):.4f} | Val Acc: {acc:.2f}%")

# 4. Ekspor Bobot & Download Hasilnya
print("\n--- Mengekspor Bobot INT8 untuk FPGA BRAM ---")
for name, param in model.named_parameters():
    export_to_mif_and_hex(param, name.replace(".", "_"))

shutil.make_archive("weights_mif", "zip", "../verilog/weights_mif")
files.download("weights_mif.zip")

Mapping Kelas: {'0_sempurna': 0, '1_penyok': 1}
Epoch [01/25] Loss: 0.2409 | Val Acc: 95.13%
Epoch [02/25] Loss: 0.1535 | Val Acc: 95.13%
Epoch [03/25] Loss: 0.1097 | Val Acc: 95.13%
Epoch [04/25] Loss: 0.0982 | Val Acc: 95.13%
Epoch [05/25] Loss: 0.0908 | Val Acc: 95.65%
Epoch [06/25] Loss: 0.0799 | Val Acc: 96.54%
Epoch [07/25] Loss: 0.0781 | Val Acc: 97.57%
Epoch [08/25] Loss: 0.0723 | Val Acc: 96.93%
Epoch [09/25] Loss: 0.0624 | Val Acc: 96.80%
Epoch [10/25] Loss: 0.0537 | Val Acc: 97.18%
Epoch [11/25] Loss: 0.0405 | Val Acc: 95.65%
Epoch [12/25] Loss: 0.0490 | Val Acc: 97.31%
Epoch [13/25] Loss: 0.0388 | Val Acc: 97.82%
Epoch [14/25] Loss: 0.0329 | Val Acc: 97.95%
Epoch [15/25] Loss: 0.0240 | Val Acc: 97.82%
Epoch [16/25] Loss: 0.0342 | Val Acc: 97.95%
Epoch [17/25] Loss: 0.0238 | Val Acc: 97.95%
Epoch [18/25] Loss: 0.0152 | Val Acc: 97.82%
Epoch [19/25] Loss: 0.0151 | Val Acc: 98.08%
Epoch [20/25] Loss: 0.0118 | Val Acc: 98.34%
Epoch [21/25] Loss: 0.0110 | Val Acc: 98.21%
Epoch [

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>